<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/06_secuencias/62_series_temporales_fisicas.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Series temporales físicas: sistema de Lorenz

**Pregunta guía:** ¿Cómo evaluamos predicción cuando el tiempo impide mezclar observaciones?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere TensorFlow.** Integramos Lorenz y predecimos $x_{t+1}$ desde
una ventana de $(x,y,z)$. La separación es cronológica: pasado para
entrenamiento, intervalo posterior para validación y futuro para test.
En un sistema caótico, error pequeño de un paso no implica trayectoria
correcta a largo plazo.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.integrate import solve_ivp
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

def lorenz(t,s,σ=10,ρ=28,β=8/3):
    x,y,z=s; return [σ*(y-x),x*(ρ-z)-y,x*y-β*z]
t=np.linspace(0,80,8001); señal=solve_ivp(lorenz,[t[0],t[-1]],[1,1,1],t_eval=t,rtol=1e-9,atol=1e-9).y.T[1000:]
n_train=int(.65*len(señal)); n_val=int(.82*len(señal))
scaler=StandardScaler().fit(señal[:n_train]); S=scaler.transform(señal)
ventana=40
X=np.array([S[i:i+ventana] for i in range(len(S)-ventana)])
y=np.array([S[i+ventana,0] for i in range(len(S)-ventana)])
idx=np.arange(len(X)); train=idx[idx+ventana<n_train]; val=idx[(idx+ventana>=n_train)&(idx+ventana<n_val)]; test=idx[idx+ventana>=n_val]


In [ ]:
persistencia=X[test,-1,0]
ridge=Ridge(alpha=1.0).fit(X[train].reshape(len(train),-1),y[train])
pred_ridge=ridge.predict(X[test].reshape(len(test),-1))
keras.utils.set_random_seed(42)
modelo=keras.Sequential([layers.Input((ventana,3)),layers.LSTM(32),layers.Dense(1)])
modelo.compile(optimizer="adam",loss="mse")
modelo.fit(X[train],y[train],validation_data=(X[val],y[val]),epochs=30,batch_size=128,shuffle=False,verbose=0,
           callbacks=[keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)])
pred_lstm=modelo.predict(X[test],verbose=0).ravel()
display(pd.DataFrame({"RMSE":[root_mean_squared_error(y[test],persistencia),root_mean_squared_error(y[test],pred_ridge),root_mean_squared_error(y[test],pred_lstm)]},index=["persistencia","Ridge","LSTM"]))


In [ ]:
tramo=slice(0,500)
plt.figure(figsize=(12,4)); plt.plot(y[test][tramo],label="real"); plt.plot(pred_lstm[tramo],label="LSTM",alpha=.8); plt.plot(persistencia[tramo],label="persistencia",alpha=.6); plt.legend(); plt.ylabel("x estandarizado"); plt.show()


**Ejercicios:** compare predicción directa a 10 pasos; genere rollout
autoregresivo; mida horizonte hasta que error supere una desviación;
compare con el tiempo de Lyapunov; explique por qué barajar ventanas antes
de separar produce una evaluación optimista.
